In [ ]:
%pip install \
    "agilerl>=2.3.5" \
    "gymnasium>=1.2.1" \
    "imageio>=2.37.0" \
    "matplotlib>=3.9.4" \
    "mpe2>=0.0.1" \
    "pettingzoo[mpe]>=1.25.0" \
    "pillow>=12.0.0" \
    "datasets"
    
# always move into /workspace (RunPod standard)
%cd /workspace

!mkdir -p RL/envs \
         RL/train \
         RL/eval \
         RL/utils \
         RL/models/MATD3 \
         RL/videos

%cd RL


### Célula 1 – Dependências e Organização das pastas de trabalho

Esta célula:

1. **Muda o diretório atual para `/workspace`**  
   Isso segue o padrão de muitos ambientes em nuvem (como RunPod), onde `/workspace` é a pasta principal de trabalho.

2. **Cria a estrutura de pastas do projeto** (se ainda não existir):
   - `RL/` – pasta raiz do projeto de Reinforcement Learning.
   - `RL/data/` – reservada para dados auxiliares (se você quiser salvar algo depois).
   - `RL/envs/` – onde ficarão os arquivos Python que definem ou encapsulam ambientes (por exemplo, `speaker_listener_env.py`).
   - `RL/models/MATD3/` – onde serão salvos os modelos treinados (checkpoints `.pt` do MATD3).
   - `RL/videos/` – pasta para armazenar vídeos dos episódios (se você decidir gravar).

3. **Entra na pasta `RL/`**  
   A partir daqui, caminho relativo como `envs/...` ou `models/MATD3/...` passa a funcionar direto.

Em resumo, esta célula só organiza o **layout de arquivos e diretórios** para o restante do código.

---

In [ ]:
%%writefile envs/speaker_listener_env.py
from mpe2 import simple_speaker_listener_v4

def make_speaker_listener_env(continuous=True, render=False):
    """
    Returns a single parallel MPE2 speaker-listener environment.
    """
    return simple_speaker_listener_v4.parallel_env(
        continuous_actions=continuous,
        render_mode="rgb_array" if render else None
    )

### Célula 2 – Definição do ambiente Speaker–Listener (MPE2)

Esta célula cria um arquivo Python (`envs/speaker_listener_env.py`) com uma função auxiliar para construir o ambiente:

- Usa a magic `%%writefile` do Jupyter para **gravar o conteúdo da célula em um arquivo `.py`** dentro da pasta `envs/`.
- Importa o ambiente `simple_speaker_listener_v4` da biblioteca **MPE2**.
- Define a função `make_speaker_listener_env(continuous=True, render=False)` que:
  - Cria uma instância **paralela** do ambiente (`parallel_env`), ou seja, capaz de rodar vários ambientes em paralelo.
  - Usa `continuous_actions=True`, permitindo usar o MATD3 (que lida com ações contínuas).
  - Configura `render_mode="rgb_array"` quando `render=True`, possibilitando capturar frames de imagem para visualização ou gravação de vídeo.

Assim, sempre que o código precisar de um ambiente Speaker–Listener, basta chamar `make_speaker_listener_env(...)` em vez de reescrever a construção do ambiente.

---

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch

# ❌ You don't need the PettingZoo import if you're using MPE2
# from pettingzoo.mpe import simple_speaker_listener_v4
# ✅ Use the new MPE2 package
from mpe2 import simple_speaker_listener_v4

from agilerl.algorithms import MATD3
from agilerl.algorithms.core.registry import HyperparameterConfig, RLParameter
from agilerl.components.multi_agent_replay_buffer import MultiAgentReplayBuffer
from agilerl.hpo.mutation import Mutations
from agilerl.hpo.tournament import TournamentSelection
from agilerl.utils.utils import (
    create_population,
    default_progress_bar,
    make_multi_agent_vect_envs,
)

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("===== AgileRL Online Multi-Agent Demo =====")

### Célula 3 – Script principal de treino e fine-tuning com MATD3

Esta é a célula mais longa e importante. Ela contém todo o script de treino do MATD3 em modo multi-agente, com otimização evolutiva de hiperparâmetros. Os principais blocos dessa célula são:

1. **Imports e configuração do dispositivo**
   - Importa `os`, `matplotlib`, `numpy`, `torch`.
   - Importa o ambiente `simple_speaker_listener_v4` (via MPE2).
   - Importa do `agilerl`:
     - o algoritmo `MATD3` (versão multi-agente),
     - `HyperparameterConfig` e `RLParameter` (para descrever hiperparâmetros mutáveis),
     - `MultiAgentReplayBuffer` (replay buffer para vários agentes),
     - `Mutations` e `TournamentSelection` (mecanismos de seleção e mutação evolutiva),
     - utilitários como `create_population`, `default_progress_bar` e `make_multi_agent_vect_envs`.
   - Define o dispositivo (`cuda` se houver GPU, caso contrário `cpu`).

In [ ]:
    # Define the network configuration
    NET_CONFIG = {
        "latent_dim": 64,
        "encoder_config": {
            "hidden_size": [64],  # Actor hidden size
        },
        "head_config": {
            "hidden_size": [64],  # Critic hidden size
        },
    }

2. **Configuração da rede neural (`NET_CONFIG`)**
   - Define tamanhos de camadas para o ator e crítico (por exemplo, camadas escondidas de tamanho 64).
   - Define `latent_dim`, isto é, o tamanho da representação interna.
   - Essas configurações são passadas para a construção das redes internas do MATD3.

In [ ]:


    # Define the initial hyperparameters
    INIT_HP = {
        "POPULATION_SIZE": 4,
        "ALGO": "MATD3",  # Algorithm
        "BATCH_SIZE": 512,  # Batch size
        "O_U_NOISE": True,  # Ornstein Uhlenbeck action noise
        "EXPL_NOISE": 0.8,  # Action noise scale
        "MEAN_NOISE": 0.0,  # Mean action noise
        "THETA": 0.15,  # Rate of mean reversion in OU noise
        "DT": 0.01,  # Timestep for OU noise
        "LR_ACTOR": 0.0001,  # Actor learning rate
        "LR_CRITIC": 0.001,  # Critic learning rate
        "GAMMA": 0.98,  # Discount factor
        "MEMORY_SIZE": 100000,  # Max memory buffer size
        "LEARN_STEP": 10,  # Learning frequency
        "TAU": 0.5,  # For soft update of target parameters
        "POLICY_FREQ": 2,  # Policy frequency
    }

3. **Hiperparâmetros iniciais (`INIT_HP`)**
   - Define parâmetros de RL como:
     - tamanho da população (`POPULATION_SIZE`),
     - `BATCH_SIZE`,
     - uso e parâmetros do ruído de Ornstein–Uhlenbeck (exploração em ações),
     - taxas de aprendizado do ator e crítico (`LR_ACTOR`, `LR_CRITIC`),
     - fator de desconto (`GAMMA`),
     - tamanho máximo do replay buffer (`MEMORY_SIZE`),
     - frequência de aprendizado (`LEARN_STEP`),
     - parâmetro `TAU` para soft-update das redes alvo,
     - `POLICY_FREQ`, que controla com que frequência o ator é atualizado em relação ao crítico.
 Obs. **Parâmetros que foram alterados para o Finetunning:**
      - `BATCH_SIZE = 128` -> `BATCH_SIZE = 512`
      - `EXPL_NOISE = 0.1` -> `EXPL_NOISE = 0.8`
      - `GAMMA = 0.95` ->  -> `GAMMA = 0.98`
      - `LEARN_STEP = 100` -> `LEARN_STEP = 10`
      - `TAU = 0.01`-> `TAU = 0.5`

---

In [ ]:
num_envs = 8

    def make_env():
        # Using MPE2 env
        return simple_speaker_listener_v4.parallel_env(continuous_actions=True)

    env = make_multi_agent_vect_envs(env=make_env, num_envs=num_envs)

    # Get agent IDs from env
    agent_ids = env.agents
    INIT_HP["AGENT_IDS"] = agent_ids  # <-- MUST happen before create_population


    # ✅ Configure the multi-agent algo input arguments as dicts keyed by agent_id
    observation_spaces = {
        agent: env.single_observation_space(agent) for agent in env.agents
    }
    action_spaces = {
        agent: env.single_action_space(agent) for agent in env.agents
    }

    # Append number of agents and agent IDs to the initial hyperparameter dictionary
    #INIT_HP["AGENT_IDS"] = env.agents

4. **Criação de ambientes vetorizados e leitura dos espaços multi-agente**
   - Define `num_envs` (por exemplo, 8), ou seja, quantos ambientes paralelos serão usados.
   - Define uma função `make_env()` que retorna uma instância `parallel_env` do `simple_speaker_listener_v4`.
   - Usa `make_multi_agent_vect_envs(...)` para criar um **vetor de ambientes multi-agente**.
   - Obtém os IDs dos agentes (`env.agents`, por exemplo `["speaker", "listener"]`) e salva em `INIT_HP["AGENT_IDS"]`.
   - Cria dicionários `observation_spaces` e `action_spaces` mapeando cada `agent_id` para seu respectivo espaço de observação e de ação.

---

In [ ]:
# Mutation config for RL hyperparameters
    hp_config = HyperparameterConfig(
        lr_actor=RLParameter(min=1e-4, max=1e-2),
        lr_critic=RLParameter(min=1e-4, max=1e-2),
        batch_size=RLParameter(min=8, max=512, dtype=int),
        learn_step=RLParameter(
            min=20, max=200, dtype=int, grow_factor=1.5, shrink_factor=0.75
        ),
    )

5. **Configuração da faixa de hiperparâmetros evolutivos (`hp_config`)**
   - Usa `HyperparameterConfig` e `RLParameter` para dizer **quais hiperparâmetros podem sofrer mutação** e em quais intervalos:
     - `lr_actor` e `lr_critic` variando em um intervalo contínuo (por exemplo `1e-4` a `1e-2`).
     - `batch_size` variando em um intervalo inteiro (por exemplo `8` a `512`).
     - `learn_step` variando em um intervalo inteiro, com fatores de crescimento e redução.
   - Isso é usado pelo módulo de mutações para fazer **Hyperparameter Optimization (HPO)** de forma evolutiva.

In [ ]:
# Create a population ready for evolutionary hyper-parameter optimisation
    pop: list[MATD3] = create_population(
        INIT_HP["ALGO"],          # algo
        NET_CONFIG,               # net_config
        INIT_HP,                  # INIT_HP (now has "AGENT_IDS")
        observation_spaces,       # observation_space
        action_spaces,            # action_space
        hp_config=hp_config,
        population_size=INIT_HP["POPULATION_SIZE"],
        num_envs=num_envs,
        device=device,
    )

6. **Criação da população de agentes MATD3**
   - Chama `create_population(...)` passando:
     - o nome do algoritmo (`"MATD3"`),
     - as configurações de rede (`NET_CONFIG`),
     - os hiperparâmetros iniciais (`INIT_HP`),
     - os espaços de observação e ação por agente,
     - `hp_config` (descrição das mutações permitidas),
     - `population_size` (tamanho da população),
     - número de ambientes (`num_envs`),
     - e o dispositivo (`device`).
   - O resultado é uma **lista de agentes MATD3**, cada um com sua própria política e hiperparâmetros, formando a população evolutiva.

---

In [ ]:
# Configure the multi-agent replay buffer
    field_names = ["obs", "action", "reward", "next_obs", "done"]
    memory = MultiAgentReplayBuffer(
        INIT_HP["MEMORY_SIZE"],
        field_names=field_names,
        agent_ids=INIT_HP["AGENT_IDS"],
        device=device,
    )

7. **Criação do replay buffer multi-agente**
   - Instancia um `MultiAgentReplayBuffer` com:
     - tamanho máximo (`MEMORY_SIZE`),
     - nomes dos campos (`["obs", "action", "reward", "next_obs", "done"]`),
     - IDs dos agentes.
   - Esse buffer armazena transições de **todos os agentes** e será usado para amostrar mini-batches durante o aprendizado.

---

In [ ]:
# Instantiate a tournament selection object (used for HPO)
    tournament = TournamentSelection(
        tournament_size=2,  # Tournament selection size
        elitism=True,  # Elitism in tournament selection
        population_size=INIT_HP["POPULATION_SIZE"],  # Population size
        eval_loop=1,  # Evaluate using last N fitness scores
    )

    # Instantiate a mutations object (used for HPO)
    mutations = Mutations(
        no_mutation=0.2,  # Probability of no mutation
        architecture=0.2,  # Probability of architecture mutation
        new_layer_prob=0.2,  # Probability of new layer mutation
        parameters=0.2,  # Probability of parameter mutation
        activation=0,  # Probability of activation function mutation
        rl_hp=0.2,  # Probability of RL hyperparameter mutation
        mutation_sd=0.1,  # Mutation strength
        rand_seed=1,
        device=device,
    )

8. **Configuração do Tournament Selection e Mutations**
   - Cria um objeto `TournamentSelection` com:
     - `tournament_size` (quantos agentes se comparam por vez),
     - `elitism=True` (para preservar sempre o melhor agente),
     - `population_size`,
     - `eval_loop` (quantas avaliações recentes considerar).
   - Cria um objeto `Mutations` que define probabilidades de:
     - não mutar,
     - mutar arquitetura de rede (ex: adicionar camada),
     - mutar parâmetros de rede,
     - mutar hiperparâmetros de RL (learning rate, batch size, etc.),
     - além de um desvio padrão (`mutation_sd`) que controla o “tamanho” da mutação.
   - Esses dois objetos implementam a parte **evolutiva** do treinamento (seleção + mutação).

---

In [ ]:
# Define training loop parameters
    max_steps = 2_000_000  # Max steps (default: 2000000)
    learning_delay = 0  # Steps before starting learning
    evo_steps = 10_000  # Evolution frequency
    eval_steps = None  # Evaluation steps per episode - go until done
    eval_loop = 1  # Number of evaluation episodes
    elite = pop[0]  # Assign a placeholder "elite" agent
    total_steps = 0

9. **Parâmetros do loop de treinamento**
   - Define valores como:
     - `max_steps` – número máximo de passos de ambiente desejado.
     - `learning_delay` – atraso antes de começar a aprender com o buffer.
     - `evo_steps` – frequência com que acontece o ciclo evolutivo (avaliação + seleção + mutação).
     - `eval_loop` – quantos episódios usar em cada avaliação.
   - Inicializa:
     - `elite` como o primeiro agente da população (será atualizado depois para o melhor),
     - um vetor `training_scores_history` para guardar a evolução das pontuações.
---

In [ ]:

    # List to store population mean scores for plotting
    training_scores_history = []

    # TRAINING LOOP
    print("Training...")
    pbar = default_progress_bar(max_steps)
    while np.less([agent.steps[-1] for agent in pop], max_steps).all():
        pop_episode_scores = []
        for agent in pop:  # Loop through population
            agent.set_training_mode(True)
            obs, info = env.reset()  # Reset environment at start of episode
            scores = np.zeros(num_envs)
            completed_episode_scores = []
            steps = 0
            for idx_step in range(evo_steps // num_envs):
                action, raw_action = agent.get_action(
                    obs=obs, infos=info
                )  # Predict action
                next_obs, reward, termination, truncation, info = env.step(
                    action
                )  # Act in environment

                scores += np.sum(np.array(list(reward.values())).transpose(), axis=-1)
                total_steps += num_envs
                steps += num_envs

                # Save experiences to replay buffer
                memory.save_to_memory(
                    obs,
                    raw_action,
                    reward,
                    next_obs,
                    termination,
                    is_vectorised=True,
                )

                # Learn according to learning frequency
                # Handle learn steps > num_envs
                if agent.learn_step > num_envs:
                    learn_step = agent.learn_step // num_envs
                    if (
                        idx_step % learn_step == 0
                        and len(memory) >= agent.batch_size
                        and memory.counter > learning_delay
                    ):
                        experiences = memory.sample(agent.batch_size)
                        agent.learn(experiences)

                # Handle num_envs > learn step; learn multiple times per step in env
                elif (
                    len(memory) >= agent.batch_size and memory.counter > learning_delay
                ):
                    for _ in range(num_envs // agent.learn_step):
                        experiences = memory.sample(agent.batch_size)
                        agent.learn(experiences)

                obs = next_obs

                # Calculate scores and reset noise for finished episodes
                reset_noise_indices = []
                term_array = np.array(list(termination.values())).transpose()
                trunc_array = np.array(list(truncation.values())).transpose()
                for idx, (d, t) in enumerate(zip(term_array, trunc_array)):
                    if np.any(d) or np.any(t):
                        completed_episode_scores.append(scores[idx])
                        agent.scores.append(scores[idx])
                        scores[idx] = 0
                        reset_noise_indices.append(idx)

                agent.reset_action_noise(reset_noise_indices)

            pbar.update(evo_steps // len(pop))

            agent.steps[-1] += steps
            pop_episode_scores.append(completed_episode_scores)

        # Evaluate population
        fitnesses = [
            agent.test(
                env,
                max_steps=eval_steps,
                loop=eval_loop,
            )
            for agent in pop
        ]
        mean_scores = [
            (
                np.mean(episode_scores)
                if len(episode_scores) > 0
                else 0
            )
            for episode_scores in pop_episode_scores
        ]
        
        # Save population mean score for plotting
        population_mean_score = np.mean(
            [score for score in mean_scores if isinstance(score, (int, float))]
        )
        training_scores_history.append(population_mean_score)

        mean_scores_display = [
            (
                score if isinstance(score, (int, float))
                else "0 completed episodes"
            )
            for score in mean_scores
        ]

        pbar.write(
            f"--- Global steps {total_steps} ---\n"
            f"Steps {[agent.steps[-1] for agent in pop]}\n"
            f"Scores: {mean_scores_display}\n"
            f"Fitnesses: {['%.2f' % fitness for fitness in fitnesses]}\n"
            f"5 fitness avgs: {['%.2f' % np.mean(agent.fitness[-5:]) for agent in pop]}\n"
            f"Mutations: {[agent.mut for agent in pop]}"
        )

        # Tournament selection and population mutation
        elite, pop = tournament.select(pop)
        pop = mutations.mutation(pop)

        # Update step counter
        for agent in pop:
            agent.steps.append(agent.steps[-1])

    

10. **Loop principal de treino**
    - Enquanto todos os agentes não atingirem `max_steps`:
      - Para cada agente na população:
        - Habilita modo de treino (`agent.set_training_mode(True)`).
        - Reseta o ambiente vetorizado (`env.reset()`).
        - Inicializa acumuladores de pontuação por ambiente (`scores`) e uma lista de `completed_episode_scores`.
        - Para um certo número de passos:
          - Usa `agent.get_action(...)` para obter ações (incluindo exploração por ruído).
          - Executa `env.step(action)` para pegar:
            - observações seguintes,
            - recompensas,
            - flags de término (`termination` / `truncation`).
          - Acumula as recompensas em `scores`.
          - Salva a transição no replay buffer (`memory.save_to_memory(...)`).
          - Se o buffer já tiver dados suficientes e o número de passos permitir, chama `agent.learn(...)` para atualizar a política usando amostras do buffer.
          - Quando algum episódio termina em um dos ambientes do vetor:
            - registra a pontuação final daquele episódio em `completed_episode_scores`,
            - reseta o acumulador de score daquele índice,
            - reseta o ruído de ação para aquele ambiente.
      - Depois de percorrer toda a população:
        - Calcula pontuações médias para cada agente.
        - Registra a **média da população** em `training_scores_history` para poder plotar depois.
        - Usa `TournamentSelection` para selecionar os agentes que continuarão (preservando o `elite`).
        - Aplica `Mutations` sobre a população selecionada para gerar novas variações de agentes.

---

In [ ]:
# Save the trained algorithm
    path = "./models/MATD3"
    filename = "MATD3_trained_agent_new.pt"
    os.makedirs(path, exist_ok=True)
    save_path = os.path.join(path, filename)
    elite.save_checkpoint(save_path)
    
    # Plot and save score evolution
    plt.figure(figsize=(12, 6))
    plt.plot(training_scores_history, linewidth=2)
    plt.title('Evolução das Pontuações Médias Durante o Treinamento', fontsize=14)
    plt.xlabel('Iterações de Evolução', fontsize=12)
    plt.ylabel('Pontuação Média da População', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    plot_path = os.path.join(path, "training_scores_evolution.png")
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"Gráfico de evolução das pontuações salvo em: {plot_path}")
    
    scores_data_path = os.path.join(path, "training_scores_history_new.npy")
    np.save(scores_data_path, np.array(training_scores_history))
    print(f"Dados das pontuações salvos em: {scores_data_path}")
    
    plt.show()

    pbar.close()
    env.close()


11. **Salvar o melhor agente (“elite”) e as curvas de treinamento**
    - Ao fim do processo:
      - Salva o agente `elite` em `./models/MATD3/MATD3_trained_agent_new.pt`.
      - Plota a lista `training_scores_history` em um gráfico de linha.
      - Salva o gráfico como `training_scores_evolution.png`.
      - Salva os dados numéricos das pontuações em `training_scores_history_new.npy`.
    - Fecha a barra de progresso e chama `env.close()` para liberar o ambiente.

Em resumo, esta célula contém **toda a lógica de treinamento**, avaliação, seleção evolutiva, mutação de hiperparâmetros, salvamento de modelo e geração de gráficos para análise posterior.

![Gráfico de evolução](../Results/models/MATD3/training_scores_evolution.png)
